# FIFA World Cup Prediction Model
## Notebook 01 — Data Collection & Exploratory Data Analysis

**Goal:** Download and understand the raw data that will power the prediction model.

### Data sources
| Dataset | File | Description |
|---|---|---|
| International match results | `data/raw/results.csv` | Every senior international match since 1872 |
| FIFA rankings | `data/raw/fifa_ranking.csv` | Monthly ranking snapshots |
| Penalty shootouts | `data/raw/shootouts.csv` | Knockout-round shootout winners |

Download links:
- **Primary (Kaggle):** https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017
- **Mirror (GitHub):** https://github.com/jfjelstul/worldcup (structured by tournament)

> Place downloaded files in `data/raw/` before running the cells below.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # make src/ importable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from src.utils import (
    load_results, load_shootouts, load_rankings,
    add_outcome_column, normalize_teams, filter_date_range,
    output_path
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# Consistent visual style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

print('All imports OK ✓')


## 1. Load & inspect raw data

In [ ]:
results = load_results()
results = normalize_teams(results)
results = add_outcome_column(results)

print(f'Total matches: {len(results):,}')
print(f'Date range:    {results.date.min().date()}  →  {results.date.max().date()}')
print(f'Unique teams:  {pd.concat([results.home_team, results.away_team]).nunique()}')
print()
results.head(8)


In [ ]:
# Data quality check
print('=== Missing values ===')
print(results.isnull().sum())
print()
print('=== Column dtypes ===')
print(results.dtypes)
print()
print('=== Tournament types ===')
print(results.tournament.value_counts().head(20))


## 2. Define the modelling window

Matches before 1990 are less predictive — squad depth, travel, and global participation
were all different. We'll use **1990–present** for feature engineering and training, but
keep the full history available for ELO initialisation.


In [ ]:
MODERN_START = '1990-01-01'

modern = filter_date_range(results, start=MODERN_START).copy()
print(f'Modern era matches: {len(modern):,}')
print(f'Date range: {modern.date.min().date()} → {modern.date.max().date()}')


## 3. Outcome distribution

In [ ]:
# Overall win/draw/loss split (neutral venues separately)
def outcome_shares(df, label):
    counts = df.outcome.value_counts(normalize=True) * 100
    print(f'{label}')
    for k in ['home', 'draw', 'away']:
        print(f'  {k:6s}: {counts.get(k, 0):.1f}%')
    print()

outcome_shares(modern, 'All modern matches')
outcome_shares(modern[modern.neutral == True],  'Neutral venue')
outcome_shares(modern[modern.neutral == False], 'Home ground')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, neutral, title in zip(
    axes,
    [False, True],
    ['Home ground', 'Neutral venue']
):
    subset = modern[modern.neutral == neutral]
    counts = subset.outcome.value_counts()[['home', 'draw', 'away']]
    colors = ['#2196F3', '#9E9E9E', '#EF5350']
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Number of matches')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{val:,}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Match outcome distribution (1990–present)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'outcome_distribution.png'), bbox_inches='tight')
plt.show()
print('Saved → outputs/outcome_distribution.png')


## 4. Goals scored analysis

Goals follow a **Poisson distribution** — this is the key statistical assumption
behind our goals-based model. We'll verify it here.


In [ ]:
from scipy.stats import poisson

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title, color in zip(
    axes,
    ['home_score', 'away_score'],
    ['Home goals per match', 'Away goals per match'],
    ['#2196F3', '#EF5350']
):
    empirical = modern[col].value_counts(normalize=True).sort_index()
    lam = modern[col].mean()
    x = np.arange(0, 9)
    poisson_pmf = poisson.pmf(x, lam)

    ax.bar(empirical.index, empirical.values, alpha=0.6, color=color,
           label=f'Observed (mean={lam:.2f})', width=0.4)
    ax.plot(x, poisson_pmf, 'ko--', ms=5, lw=1.5, label=f'Poisson(λ={lam:.2f})')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Goals')
    ax.set_ylabel('Probability')
    ax.legend()
    ax.set_xlim(-0.5, 8.5)

plt.suptitle('Goals distribution vs Poisson fit', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'goals_poisson_fit.png'), bbox_inches='tight')
plt.show()

print(f"Home avg goals: {modern.home_score.mean():.3f}")
print(f"Away avg goals: {modern.away_score.mean():.3f}")
print(f"Home advantage: {modern.home_score.mean() - modern.away_score.mean():.3f} goals/match")


In [ ]:
yearly = modern.copy()
yearly['year'] = yearly.date.dt.year
yearly_stats = yearly.groupby('year').agg(
    avg_home=('home_score', 'mean'),
    avg_away=('away_score', 'mean'),
    matches=('home_score', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.plot(yearly_stats.year, yearly_stats.avg_home, 'b-o', ms=4, lw=1.8, label='Home goals')
ax1.plot(yearly_stats.year, yearly_stats.avg_away, 'r-o', ms=4, lw=1.8, label='Away goals')
ax1.fill_between(yearly_stats.year, yearly_stats.avg_home, yearly_stats.avg_away, alpha=0.08, color='blue')
ax1.set_xlabel('Year')
ax1.set_ylabel('Average goals per match')
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.bar(yearly_stats.year, yearly_stats.matches, alpha=0.15, color='gray', label='# matches')
ax2.set_ylabel('Number of matches')
ax2.legend(loc='upper right')

plt.title('Average goals per match over time', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'goals_over_time.png'), bbox_inches='tight')
plt.show()


## 5. World Cup–specific analysis

In [ ]:
wc = modern[modern.tournament == 'FIFA World Cup'].copy()
wc['year'] = wc.date.dt.year

print(f'World Cup matches (modern era): {len(wc):,}')
print(f'Tournaments covered: {sorted(wc.year.unique())}')
print()

wc_by_year = wc.groupby('year').agg(
    matches=('home_score', 'count'),
    avg_total_goals=('home_score', lambda x: (x + wc.loc[x.index, 'away_score']).mean()),
    home_wins=('outcome', lambda x: (x == 'home').mean()),
    draws=('outcome',    lambda x: (x == 'draw').mean()),
    away_wins=('outcome', lambda x: (x == 'away').mean()),
).reset_index()

wc_by_year


In [ ]:
# Most successful World Cup teams
home_wins = wc[wc.outcome == 'home'].groupby('home_team').size()
away_wins = wc[wc.outcome == 'away'].groupby('away_team').size()
total_wins = (home_wins.add(away_wins, fill_value=0)
              .sort_values(ascending=False)
              .head(15)
              .astype(int))

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#FFD700' if i < 3 else '#2196F3' for i in range(len(total_wins))]
bars = ax.barh(total_wins.index[::-1], total_wins.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Total knockout/group stage wins')
ax.set_title('Most wins in World Cup history (modern era)', fontweight='bold')
for bar, val in zip(bars, total_wins.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'wc_wins_by_team.png'), bbox_inches='tight')
plt.show()


## 6. Head-to-head helper (preview)

In [ ]:
def head_to_head(df, team_a, team_b, since='2010-01-01'):
    """Return all matches between two teams since a given date."""
    mask = (
        ((df.home_team == team_a) & (df.away_team == team_b)) |
        ((df.home_team == team_b) & (df.away_team == team_a))
    )
    subset = df[mask & (df.date >= since)].copy()
    subset['label'] = subset.apply(
        lambda r: f"{r.home_team} {int(r.home_score)}–{int(r.away_score)} {r.away_team}", axis=1
    )
    return subset[['date', 'label', 'tournament', 'outcome']].reset_index(drop=True)

# Example
head_to_head(modern, 'Brazil', 'Argentina')


## 7. Save processed dataset

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', 'matches_modern.csv')
modern.to_csv(processed_path, index=False)
print(f'Saved {len(modern):,} rows → {processed_path}')

print()
print('=== Column summary ===')
print(modern.dtypes)
print()
print('Sample row:')
modern.tail(1).T


## ✅ EDA complete — what we learned

| Finding | Implication for modelling |
|---|---|
| Home advantage adds ~0.3 goals/match | Include `is_neutral` as a feature |
| Goals are Poisson-distributed | Use Poisson regression for goal prediction |
| Modern era (1990+) is most relevant | Train on 1990–present |
| Strong teams win consistently | ELO ratings will be powerful features |

**Next notebook → `02_features.ipynb`:** Build ELO ratings, recent form, and head-to-head features.
